# 4~5주차: 논문 마무리

## 금주 학습목표
* 논문 1회독
    * Data, software, hardware
    * Methodology
    * Results
    * Conclusion
* 파트 배분

## 1. 논문 이해(Data, software, hardware)
* Data
    * 데이터 선정 S&P500
    * Survivor basis 제거를 위해 S&P 500에서 제외되거나 합쳐진 주식을 포함하여 하나의 matrix로 정리
    * 주가가 올랐는지 내렸는지 나타내도록 binary matrix로 정리
* software
    * Python
    * Tensorflow
* hardware
    * NVDA GPU

## 2. 논문 이해(Methodology)
* Training set 생성
    * Training set을 size를 1000개로 잡아서 750개와 250개로 쪼깸
    * 750개의 데이터를 한 블록으로 삼아 240일을 윈도우로 잡고 롤링 윈도우 실행
    * 블럭이 끝나면 250일만큼 전체 블록을 밀고 다시 수행
    * 총 23개의 훈련세트와 38만개 가량의 시퀀스 생성됨
* Feature 설정
    * 유일한 입력 Feature로 주식의 1일 단순 수익률을 사용함
    * 단순 가격이 아닌 수익률을 사용하는 이유: 주식마다 주가 단위와 스케일이 달라서 발생하는 학습 오류를 방지하기 위함임
    * 정규화: 750일 훈련 기간 동안 S&P500 전체 종목이 기록한 수익률을 한데 묶어 거대한 벡터로 만든 뒤, 전체 평균을 빼고 전체 표준편차로 나눔
    * 개별 종목이 아닌 전체 종목 기준으로 정규화하는 이유: 개별 종목별로 정규화하면 모든 주식의 변동성이 똑같아 보임. 전체 잣대로 나누어주어야 인공지능이 어떤 주식이 남들보다 유독 널뛰기가 심한지 종목 간의 상대적 특성을 잃지 않고 학습할 수 있음
* Target 설정
    * 예측해야 하는 t+1시점의 S&P500 전체 주식 수익률을 내림차순 정렬하여 정확히 가운데 있는 중간값을 도출함
    * 개별 주식의 익일 수익률이 이 중간값 이상이면 1, 미만이면 0으로 이진 분류 타겟을 라벨링함
    * 0보다 크면 오름, 작으면 내림으로 단순 분류하지 않는 이유: 시장 전체가 폭락하는 날에도 상대적으로 덜 떨어질 방어주를 찾고, 폭등장에서도 남들보다 더 오를 주도주를 골라내기 위함. 즉, 시장 방향에 휘둘리지 않는 시장 중립적 롱숏 전략을 구사하기 위한 세팅
* LSTM의 이해
    * LSTM은 RNN의 일종으로, 기존 RNN의 치명적 결함인 단기기억 한계(기울기 소실 등)를 극복하여 Long-term dependencies을 학습하는 데 특화된 모델임
    * 은닉층 내부에 3개의 게이트(망각, 입력, 출력)를 두어 Cell state의 흐름을 조절함
    * 망각 게이트: 과거의 기억 중 불필요한 노이즈를 지움.
    * 입력 게이트: 새롭게 들어온 정보 중 유효한 값을 기억에 추가함
    * 출력 게이트: 기존값으로부터 예측값 돛ᅟᅮᆯ
    * 출력 게이트가 따로 존재하는 이유: 장기 기억(s)이 망각/입력 관문을 거쳐 훌륭하게 최적화되어 있더라도, 그 방대한 장기 기억 중 현재 시점의 주가 예측 텐서로 당장 써먹을 단기 기억(h)만 밖으로 빼내어 분리하기 위한 출구가 필요하기 때문임
    * RMSprop 옵티마이저: 가중치를 업데이트하며 오차를 줄이는 알고리즘. 과거의 기울기는 잊고 최근 기울기를 강하게 반영하여 파라미터별 학습률을 조절하므로, 노이즈가 심한 주가 데이터 학습에 매우 탁월함
    * 과적합 방지를 위해 순환 레이어 입력의 10%를 무작위로 비활성화하는 Dropout과, 검증 세트의 오차가 10번 이상 개선되지 않으면 과거 최적 모델로 복구하여 학습을 멈추는 Early stopping를 적용함
* Benchmark 모델
    * 메모리가 없는 통상적인 분류 방법들이 시계열 데이터에서 얼마나 한계가 있는지 대조하기 위해 3가지 벤치마크를 설정함
    * ROF: 원본 데이터에서 복원 추출하여 만든 수많은 부트스트랩 샘플에 각각 의사결정 나무를 학습시키고 다수결로 예측하는 앙상블 기법. 노이즈에 강해 튜닝 없이도 좋은 성과를 내는 훌륭한 대조군임
    * DNN: LSTM의 구조적 우월성을 증명하기 위해 세팅한 표준적인 피드포워드 딥러닝 모델
    * LOG: 베이스라인이 되는 기초 통계 분류기. 과적합을 막기 위해 가중치 크기에 벌점을 부여하는 L2 Regularization을 교차 검증으로 최적화하여 적용
* 예측, 순위 매기기 및 거래
    * 학습을 마친 모델들이 내일 특정 주식이 시장 중간값을 상회할 확률을 0~100% 사이의 값으로 예측하여 반환함
    * 매일 500여 개의 주식을 이 예측 확률이 높은 순서대로 1등부터 꼴찌까지 내림차순 정렬함
    * 확률 상위 k개 종목은 매수하고, 하위 k개 종목은 공매도하는 양방향 포트폴리오를 매일 기계적으로 구축하여 거래
    * Long-Short 및 차익거래: 오를 주식은 사고 내릴 주식은 빌려서 파는 전략으로, 거시 경제가 붕괴하여 주식 시장 전체가 하락하더라도 매도 포지션에서 수익이 나 포트폴리오의 시장 위험을 완전히 헷징하는 안정적 투자 기법임

## 3. 논문 이해(Result)
* Overview
    * 포트폴리오 크기(k=10, 50, 100, 150, 200)를 다양하게 설정하여 모델 간의 성과를 비교함
    * 대부분의 k 수준에서 LSTM이 일일 평균 수익률, 샤프 비율, 예측 정확도 면에서 RAF, DNN, LOG을 압도
    * 포트폴리오 크기가 커질수록 수익률은 낮아지는 경향을 보였으며, 수익성과 모델 간 성능 차이가 가장 극명하게 나타나는 k=10 (상위 10종목 매수, 하위 10종목 공매도) 포트폴리오에 이후 분석을 집중함
* 예측성능
    * Diebold-Mariano 검정: 두 모델을 직접 맞붙여 예측 오차의 차이가 통계적으로 유의미한지 확인하는 테스트. LSTM의 예측력이 다른 모든 모델보다 명백하게 우수하다는 것을 통계적으로 입증함
    * Pesaran-Timmermann 검정: 모델의 예측 방향(상승/하락)과 실제 주가 방향이 동전 던지기처럼 무관한지(독립적인지) 확인하는 테스트. p-value가 0에 수렴하여 모든 머신러닝 모델의 예측이 단순한 우연이 아님을 증명
    * p-value의 의미: 관찰된 결과가 우연히 일어났을 확률. 보통 0.05 미만이면 우연이 아닌 진짜 실력으로 인정함
    * 원숭이가 무작위로 다트를 던져 종목을 뽑는 10만 번의 시뮬레이션 결과와 비교했을 때, LSTM이 달성한 54.3%의 분류 정확도가 우연히 발생할 확률은 거의 0에 수렴하여 압도적 우위를 확인
* Financial Performance
    * 현실적인 미국 대형주 거래 비용인 편도 0.05%의 수수료를 매매 시마다 차감한 후에도 LSTM은 일 평균 0.26%, 연환산 82.29%의 높은 순수익률을 달성
    * Newey-West t-통계량: 주식 데이터처럼 어제의 가격이 오늘에 영향을 주는 자기상관성과 변동성이 변하는 이분산성을 수학적으로 교정하여 오차의 거품을 빼고 엄격하게 채점하는 지표. 임계값(1.96)을 훌쩍 넘는 9.5792를 기록하여 수익 창출이 통계적으로 확실한 실력임을 증명
    * 위험 1단위당 수익을 나타내는 Sharpe Ratio 2.34로 벤치마크 모델 중 최고 수준을 기록해 수익성과 위험 관리 능력을 동시에 입증
* 시간 경과에 따른 LSTM
    * 1993~2000년: LSTM과 딥러닝 개념이 시장에 널리 알려지거나 하드웨어적으로 구현되기 전이었으므로 비정상적으로 높은 초과 수익을 독식함
    * 2001~2009년: 전반적인 수익률은 점차 감소했으나, 2008년 글로벌 금융 위기 당시 패닉 셀링으로 주가가 왜곡되며 차익거래 기회가 발생. 이때 이상치와 노이즈에 강한 앙상블 기법인 RAF가 일시적으로 LSTM의 성과를 능가
    * 2010~2015년: 헤지펀드 산업에 알고리즘 트레이딩이 보편화되며 기계학습이 찾던 초과 수익 기회가 구조적으로 소멸(Arbitraged away)함. RAF는 마이너스 수익으로 돌아섰고, LSTM도 거래 비용 차감 후 수익률이 0 근처에서 횡보하게 됨
* 모델 선정 top flop 주식
    * 인공지능의 블랙박스 내부를 열기 위해, LSTM이 매수와 공매도로 강하게 추천한 종목들의 과거 240일 궤적을 겹쳐서 시각적 및 통계적으로 역추적
    * 지지부진한 모멘텀과 단기 반전: 모델이 선택한 주식들은 과거 230일 동안은 시장 평균보다 수익률이 저조한 소외주들이었음. 그러나 예측 직전 10일 동안 수직으로 폭락하거나 폭등한 뒤 다음날 반대로 튕겨 오르는 Short-term reversal 현상을 강력하게 띠고 있었음
    * Leptokurtic, 고첨 분포: 정규분포보다 중심이 뾰족하고 양쪽 꼬리가 두꺼운 분포. LSTM이 선택한 주식들은 평소엔 잔잔하다가 극단적인 폭등/폭락 두꺼운 꼬리가 자주 발생하는 위험한 주식들이었음
* 수익성의 원동력
    * LSTM의 투자 철학을 모방하여 최근 5일간 가장 많이 떨어진 주식을 사고, 가장 많이 오른 주식을 판다는 인간의 단순 규칙 기반 전략을 테스트
    * 분석 결과, 단순 전략은 거래 비용 차감 전 일 수익률 0.23%로 LSTM 성과(0.46%)의 딱 절반에 그침. 이는 인공지능이 표면적 규칙 하나만 본 것이 아니라, 인간이 파악하기 힘든 복잡한 비선형적 상호작용을 통해 남은 50%의 숨은 수익을 끌어냈음을 의미함
    * 파마-프렌치 다요인 검증: 인공지능이 낸 수익이 널리 알려진 위험에 기대어 낸 뻔한 수익인지 회귀 분석
* 시장에 대한 강건성
    * Bid-ask bounce 왜곡: 폭락한 주식을 살 때는 비싼 매도 호가를 내야 하므로, 단순히 장 마감 종가로 매매했다고 가정하면 백테스트 수익률이 실제보다 부풀려지는 현상
    * 이러한 미시구조적 착시를 없애기 위해 거래량 가중평균 가격을 체결 단가로 적용하여 보수적으로 재평가함
    * 또한 잦은 매매 수수료를 줄이기 위해 5일을 보유하는 주간 모델을 도입함. 이때 특정 요일에 진입하여 생기는 운을 없애기 위해 5개의 Overlapping 포트폴리오를 하루씩 Offset를 두고 굴려 평균을 냄
    * 일시적 호가 거품을 거르기 위해 신호 발생 후 24시간을 대기하는 One-day waiting이라는 가혹한 스트레스 테스트를 적용함
    * 이 모든 현실적 마찰 비용과 페널티를 적용하고도 연환산 20~40%대의 경제적/통계적으로 유의미한 초과 수익이 살아남아 모델의 강건성이 최종 입증됨

## 4. 논문 이해(Conclusion)
* 금융 시계열 데이터처럼 노이즈가 극심하고 시간의 흐름이 중요한 분야에서, 장기 기억을 조절하는 LSTM 아키텍처가 메모리 없는 모델(ROF, DNN, LOG)보다 예측력 및 수익률에서 본질적으로 가장 우월한 최선의 선택지임을 대규모 실증 연구로 입증함
* 통계 분석을 통해 인공지능의 블랙박스 내부를 성공적으로 해체함. 인간이 사전 지식을 주입하지 않았음에도, 모델이 오직 순수한 숫자 시퀀스만으로 자본 시장의 이상 현상인 단기 반전과 고변동성 선호 패턴을 독학으로 터득했음을 밝혀냄
* 인공지능이 발견한 가장 뚜렷한 특징을 인간의 1차원적인 규칙 기반 전략으로 모방했을 때 LSTM 성과의 절반밖에 도달하지 못함. 이는 딥러닝이 선형적 공식이나 전통적 위험 요인(파마-프렌치 모델 등)으로는 측정할 수 없는 매우 고차원적인 비선형 상호작용의 결합, 즉 순수 Alpha를 창출하고 있음을 과학적으로 증명함
* VWAP 기반 체결, 주간 포트폴리오 도입, 하루 지연 주문, 편도 5bps 거래 비용 차감 등 실전 매매에서 겪는 가혹한 미시구조적 마찰 제약을 겹겹이 두르고도 성과가 무너지지 않아 전략의 실질적인 Robustness을 확인받음
* 하지만 2010년 이후 초과 수익률이 0에 수렴하는 현상을 통해, 아무리 탁월한 딥러닝 패턴이라도 알고리즘 트레이딩이 보편화되고 시장에 기회가 노출되면 결국 차익거래를 통해 수익이 소멸하고 만다는 효율적 시장 가설의 냉혹한 한계를 동시에 시사함